# 02 — Extract Real Dataset (MediaPipe pose → per-rep features)

Walks your real dataset as laid out: `data/<exercise name>/*.mp4 | *.MOV` (e.g. `data/squat/squat_1.MOV`) — this is the raw Kaggle "Workout/Exercises Video" folder structure. Each folder name IS the exercise label; **there is no safe/unsafe column anywhere in the source data.**

For each video:
1. Runs MediaPipe Pose over every frame (`biomechanics.py`)
2. Segments completed reps via the shared `RepSegmenter`
3. Aggregates each rep into the mean/std/range feature vector the models train on
4. Also saves a fixed-length (30-step) resampled per-frame sequence per rep to `real_sequences.npz`, for a genuine (non-reconstructed) CNN-BiLSTM input
5. Falls back to treating a whole video as one rep if the segmenter never completes a full cycle (common on unusual camera angles)

**Progress is checkpointed** — already-processed videos are skipped on re-run (tracked by filename in the output CSV), so you can stop and resume freely.

**IMPORTANT — mediapipe version:** mediapipe releases from roughly late 2025 onward dropped the legacy `mp.solutions.pose` API entirely, and the in-between versions that still had it have since been pulled from PyPI — there is no installable version left with that API. This notebook uses the current **Tasks API** instead, via `pose_backend.py` (see that module's docstring). The first run downloads a small model file (~30MB) automatically.

After extraction, labels are assigned via `label_rules.py`'s automatic heuristic — **there is no expert ground truth in this dataset**; this is a documented, transparent stand-in (see that module's docstring for the full justification). Pass a real `labels_csv` once you have any expert-labelled videos (see `data/raw/LABELLING_PROTOCOL.md`) and it will override the heuristic for those videos.

In [ ]:
import sys
sys.path.append('.')

import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from biomechanics import (compute_frame_features, RepSegmenter,
                           aggregate_rep_features, FEATURE_COLUMNS)
from label_rules import auto_label, merge_with_manual_labels
from pose_backend import PoseDetector

In [ ]:
# ---- CONFIG -- edit these for your setup ----
DATA_ROOT = Path("../data")
EXERCISES = ["squat", "deadlift", "romanian deadlift"]
OUT_CSV = Path("../data/processed/rep_features_real.csv")
OUT_SEQ = Path("../data/processed/real_sequences.npz")
LABELS_CSV = None  # e.g. "../data/raw/labels.csv" once you have expert labels
MAX_VIDEOS = None  # set e.g. 5 for a quick smoke test before running the full set

SEQ_STEPS = 30
SEQ_FEATURE_BASES = ["knee_angle_L", "knee_angle_R", "knee_angle_mean",
                      "hip_hinge_angle", "trunk_lean_angle", "knee_symmetry"]

In [ ]:
def resample_sequence(rep_frames, n_steps: int = SEQ_STEPS) -> np.ndarray:
    """Linearly resample a variable-length list of per-frame feature dicts to a
    fixed n_steps x n_features array -- real per-frame data for the CNN-BiLSTM,
    replacing the pseudo-sequence reconstruction used with synthetic data."""
    n_frames = len(rep_frames)
    src_t = np.linspace(0, 1, n_frames)
    tgt_t = np.linspace(0, 1, n_steps)
    seq = np.zeros((n_steps, len(SEQ_FEATURE_BASES)))
    for i, base in enumerate(SEQ_FEATURE_BASES):
        values = np.array([f[base] for f in rep_frames], dtype=float)
        seq[:, i] = np.interp(tgt_t, src_t, values)
    return seq


def extract_reps_from_video(video_path: Path, pose: PoseDetector):
    """Run MediaPipe over every frame, segment reps, return (agg_features, raw_sequence)
    tuples -- one per completed rep. Falls back to the whole video as one rep if the
    segmenter never completes a cycle."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  WARNING: could not open {video_path.name}")
        return []

    segmenter = RepSegmenter()
    completed = []
    all_frames = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        landmarks = pose.detect(rgb)
        if landmarks is None:
            continue
        feats = compute_frame_features(landmarks)
        all_frames.append(feats)
        rep = segmenter.update(feats["knee_angle_mean"], feats)
        if rep is not None and len(rep) >= 5:
            completed.append(rep)

    cap.release()
    if not completed and len(all_frames) >= 8:
        completed.append(all_frames)  # fallback: whole video as one rep

    return [(aggregate_rep_features(r), resample_sequence(r)) for r in completed]


def find_videos(data_root: Path, exercise: str):
    folder = data_root / exercise
    if not folder.exists():
        return []
    exts = {".mp4", ".mov", ".mkv", ".avi"}
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in exts)

In [ ]:
# ---- resume support: skip videos already in the output CSV ----
already_done = set()
rows, seqs = [], []

if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    already_done = set(prev["source_video"].unique())
    rows = prev.to_dict("records")
    if OUT_SEQ.exists():
        seqs = list(np.load(OUT_SEQ)["sequences"])

videos = []
for ex in EXERCISES:
    vids = find_videos(DATA_ROOT, ex)
    print(f"{ex}: found {len(vids)} videos")
    videos.extend((ex, v) for v in vids)

todo = [(ex, v) for ex, v in videos if v.name not in already_done]
if MAX_VIDEOS:
    todo = todo[:MAX_VIDEOS]
print(f"\n{len(already_done)} videos already processed (resuming), {len(todo)} to go this run.\n")

In [ ]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with PoseDetector() as pose:
    for i, (exercise, video_path) in enumerate(todo, 1):
        t0 = time.time()
        reps = extract_reps_from_video(video_path, pose)
        dt = time.time() - t0
        print(f"[{i}/{len(todo)}] {exercise}/{video_path.name}: {len(reps)} reps ({dt:.1f}s)")

        for agg, seq in reps:
            agg["exercise"] = exercise
            agg["source_video"] = video_path.name
            rows.append(agg)
            seqs.append(seq)

        # checkpoint + label after every video, so an interrupted run is still usable
        out_df = pd.DataFrame(rows)
        cols = FEATURE_COLUMNS + ["exercise", "source_video"]
        out_df = out_df[[c for c in cols if c in out_df.columns]]
        out_df = auto_label(out_df)
        if LABELS_CSV:
            out_df = merge_with_manual_labels(out_df, LABELS_CSV)
        out_df.to_csv(OUT_CSV, index=False)
        if seqs:
            np.savez(OUT_SEQ, sequences=np.stack(seqs))

print("\nDone.")

In [ ]:
final_df = pd.read_csv(OUT_CSV)
print(f"{len(final_df)} rep rows from {final_df['source_video'].nunique()} videos")
print(final_df["label"].value_counts(normalize=True).rename("class_balance"))
print()
print(final_df.groupby("exercise")["label"].value_counts())
final_df.head()